<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/09-user-defined-functions/02-pandas-function-api.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 9.2 — Pandas Function API: applyInPandas, mapInPandas, cogroup

Whole pandas DataFrames in, any DataFrame out. We detrend per
customer, stream-transform partitions with changed row counts, and
do a merge_asof via cogroup — plus the memory contract that makes
these the easiest APIs in Spark to OOM.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("9.2-pandas-function-api")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)
orders.select("order_id", "customer_id", "order_date", "revenue").show(5)

## applyInPandas: one whole group per call

Spark shuffles by the key, hands your function each group as a
pandas DataFrame, and stitches the returned frames together under
the schema you declare. Building the schema from the input plus the
new column keeps the type contract honest.

In [ ]:
import pandas as pd

subset = orders.select("order_id", "customer_id", "order_date", "revenue")
out_schema = subset.schema.add("rev_vs_cust_mean", "double")

def detrend(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("order_date")
    pdf["rev_vs_cust_mean"] = (pdf["revenue"] - pdf["revenue"].mean()).round(2)
    return pdf

(subset.groupBy("customer_id")
 .applyInPandas(detrend, out_schema)
 .show(8))

In [ ]:
# need the grouping key inside? two-argument form: f(key, pdf).
# note std() on a 1-order group is NaN -> null. Decide; don't discover.
def group_profile(key, pdf):
    (country,) = key
    return pd.DataFrame([{
        "country": country,
        "orders": len(pdf),
        "rev_total": round(float(pdf["revenue"].sum()), 2),
        "rev_std": float(pdf["revenue"].std()),
    }])

(orders.groupBy("country")
 .applyInPandas(group_profile,
                "country string, orders long, rev_total double, rev_std double")
 .show())

## The memory contract

That was 41 rows. The same code groups 2 TB by `country` into ~12
pandas DataFrames — each one entirely inside one Python worker's
memory. applyInPandas + low-cardinality or skewed keys (7.4) = the
canonical Python-worker OOM. Check the Exchange in the plan: same
shuffle currency as every groupBy.

In [ ]:
subset.groupBy("customer_id").applyInPandas(detrend, out_schema).explain()
# Exchange hashpartitioning(customer_id) -> FlatMapGroupsInPandas

## mapInPandas: partition streaming

No shuffle, no groups: an iterator of Arrow-sized batches per
partition. Never holds a partition in memory, and each batch may
yield any number of rows — filter, expand, or both in one pass.

In [ ]:
def parse_and_filter(batches):
    # expensive setup would go here — once per task (9.1's iterator trick)
    for pdf in batches:
        good = pdf[pdf["quantity"].notna()].copy()
        good["gross"] = (good["quantity"] * good["unit_price"] * 1.19).round(2)
        yield good[["order_id", "gross"]]

out = orders.mapInPandas(parse_and_filter, "order_id int, gross double")
print(orders.count(), "->", out.count())     # row count changed: 41 -> 39
out.show(5)

## cogroup: merge_asof across two tables

Per key, your function gets BOTH sides' groups. Classic use: join
each order to the exchange rate in force ON its date — a
"most recent earlier row" join plain Spark can't express cleanly.
Keys on one side only still fire with an empty frame — handle it.

In [ ]:
rates = spark.createDataFrame(
    [("DE", "2026-05-20", 1.08), ("DE", "2026-06-03", 1.10),
     ("US", "2026-05-25", 1.00), ("GB", "2026-06-01", 0.85)],
    "country string, valid_from string, rate double")

def with_rate(orders_pdf, rates_pdf):
    orders_pdf = orders_pdf.copy()
    orders_pdf["order_date"] = pd.to_datetime(orders_pdf["order_date"])
    orders_pdf = orders_pdf.sort_values("order_date")
    if rates_pdf.empty:                       # null-country group, IN, ...
        orders_pdf["rate"] = float("nan")
    else:
        rates_pdf = rates_pdf.copy()
        rates_pdf["valid_from"] = pd.to_datetime(rates_pdf["valid_from"])
        rates_pdf = rates_pdf.sort_values("valid_from")
        orders_pdf = pd.merge_asof(
            orders_pdf, rates_pdf[["valid_from", "rate"]],
            left_on="order_date", right_on="valid_from"
        ).drop(columns="valid_from")
    orders_pdf["order_date"] = orders_pdf["order_date"].dt.strftime("%Y-%m-%d")
    return orders_pdf[["order_id", "country", "order_date", "rate"]]

(orders.select("order_id", "country", "order_date")
 .groupBy("country")
 .cogroup(rates.groupBy("country"))
 .applyInPandas(with_rate,
                "order_id int, country string, order_date string, rate double")
 .orderBy("country", "order_date")
 .show(12))
# orders dated before their country's first rate get null — merge_asof
# found no "earlier row". An exact-date join would have nulled far more.

## Spark 4.0: the Arrow-native twins

Same shapes, pyarrow Tables instead of pandas — no pandas
conversion cost, no NaN rules. Reach for them when pandas itself is
the bottleneck.

In [ ]:
# mapInArrow: batches arrive as pyarrow.RecordBatch
# import pyarrow as pa
# def per_batch(batches):
#     for batch in batches:
#         yield batch                      # transform with pyarrow.compute
# orders.mapInArrow(per_batch, orders.schema).count()
# (groupBy().applyInArrow(...) is the applyInPandas twin, Spark 4.0+)
print("see also: applyInArrow / mapInArrow in the 4.0 docs")

## Exercises

1. Rewrite `parse_and_filter` as native `where` + `withColumn`.
   Compare plans. What would have to be true of the per-row logic
   for mapInPandas to be the right call anyway?
2. Add a `rank_in_customer` column (1 = biggest revenue) inside
   `detrend` with pandas, then produce identical output with a
   Module-8 window. Which wins, and on what grounds (plan, memory,
   readability)?
3. Group the FX example by `order_date` instead of `country` —
   why is that nonsense for merge_asof? What does the grouping key
   have to mean for cogroup-asof to be correct?
4. Simulate the whale: build a skewed key with `spark.range` +
   `when` (7.4 style) and applyInPandas over it. Watch task
   durations in the UI. Then fix it WITHOUT abandoning
   applyInPandas (hint: composite key, two passes).
5. Your team wants applyInPandas(f, key=customer_id) where f only
   computes `pdf["revenue"].sum()`. Argue them out of it in two
   sentences, with the replacement.

In [ ]:
# your exercise code here